# Notebook 05 - Concept transfer (Derm7pt -> DDI) + powered fairness with CIs

DDI / PAD have no 7-point concept ground truth, so the zero-shot concepts were weak. Here we **train supervised concept probes on Derm7pt** (which has concept labels) and **transfer** them to DDI embeddings, then redo the fairness analysis with **bootstrap confidence intervals**.

- **Anchor:** DermLIP zero-shot malignancy AUROC by skin tone on biopsy-proven DDI (with CIs).
- **Compare:** zero-shot vs transferred concepts as the bottleneck, ERM vs Group-DRO, per skin tone.

Inputs (local): `derm7pt_dermlip_cache/features.npz` (Notebook A) and `fairness_cache/features_ddi.npz` (Notebook C1). CPU is fine.

In [ ]:
# --- 1. Load Derm7pt (concept GT) + DDI caches ---
import os, glob, json
import numpy as np
import torch

def find_one(name):
    for r in ['/kaggle/input', '.', '..']:
        h = sorted(glob.glob(os.path.join(r, '**', name), recursive=True))
        if h:
            return h[0]
    return None

d7 = np.load(find_one('features.npz'), allow_pickle=True)
E7 = torch.tensor(d7['emb_derm'], dtype=torch.float32)
G7 = torch.tensor(d7['gt_concepts'], dtype=torch.float32)
sp7 = d7['split'].astype('U8')
CONCEPTS = ['atyp_pigment_net', 'blue_white_veil', 'atyp_vascular', 'irreg_streaks', 'irreg_pigment', 'irreg_dots', 'regression']

dz = np.load(find_one('features_ddi.npz'), allow_pickle=True)
Eddi = torch.tensor(dz['emb'], dtype=torch.float32)
Czs = torch.tensor(dz['concept_scores'], dtype=torch.float32)
yddi = torch.tensor(dz['malignant'].astype(np.float32))
gddi = dz['group'].astype('U8')
mprob = dz['malig_prob']
GROUPS = ['light', 'mid', 'dark']
print('derm7pt emb', tuple(E7.shape), '| ddi emb', tuple(Eddi.shape))

In [ ]:
# --- 2. Metrics + bootstrap CI ---
def auc(y, s):
    y = np.asarray(y).astype(float); s = np.asarray(s).astype(float)
    m = ~np.isnan(s); y = y[m]; s = s[m]
    pos = s[y == 1]; neg = s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float('nan')
    a = np.concatenate([pos, neg]); r = a.argsort().argsort().astype(float) + 1
    return (r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))

def boot_ci(y, s, n=500, seed=0):
    y = np.asarray(y); s = np.asarray(s); rng = np.random.default_rng(seed)
    idx = np.arange(len(y)); out = []
    for _ in range(n):
        b = rng.choice(idx, len(idx), replace=True)
        a = auc(y[b], s[b])
        if not np.isnan(a):
            out.append(a)
    if not out:
        return (float('nan'), float('nan'), float('nan'))
    return (float(np.mean(out)), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5)))

def fwd(C, P):
    W1, b1, W2, b2 = P
    return (torch.relu(C @ W1 + b1) @ W2 + b2).squeeze(1)

def monotonicity(P, C):
    if len(C) == 0:
        return float('nan')
    ok = []
    for i in range(C.shape[1]):
        v1 = C.clone(); v1[:, i] = 1.0
        v0 = C.clone(); v0[:, i] = 0.0
        ok.append((torch.sigmoid(fwd(v1, P)) >= torch.sigmoid(fwd(v0, P)) - 1e-6).float().mean().item())
    return float(np.mean(ok))

In [ ]:
# --- 3. Train concept probes on Derm7pt, transfer to DDI ---
tr7 = sp7 == 'train'; te7 = sp7 == 'test'

def train_probe(iters=1500, lr=0.05):
    torch.manual_seed(0)
    W = torch.zeros(E7.shape[1], 7, requires_grad=True)
    b = torch.zeros(7, requires_grad=True)
    opt = torch.optim.Adam([W, b], lr=lr)
    Xtr, Ytr = E7[tr7], G7[tr7]
    for it in range(iters):
        loss = torch.nn.functional.binary_cross_entropy_with_logits(Xtr @ W + b, Ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    return W.detach(), b.detach()

Wp, bp = train_probe()
probe_te = torch.sigmoid(E7[te7] @ Wp + bp).numpy()
print('Derm7pt test concept AUROC (supervised probe):')
for i, n in enumerate(CONCEPTS):
    print('  {0:18s} {1:.3f}'.format(n, auc(G7[te7][:, i].numpy(), probe_te[:, i])))
Cxfer = torch.sigmoid(Eddi @ Wp + bp)
print('DDI concept mean -- transfer:', round(float(Cxfer.mean()), 3), '| zero-shot:', round(float(Czs.mean()), 3))

In [ ]:
# --- 4. ANCHOR: DermLIP zero-shot malignancy disparity by skin tone (DDI), with CIs ---
print('DermLIP zero-shot malignancy AUROC by skin tone (DDI, biopsy-proven):')
for grp in GROUPS:
    m = gddi == grp
    a, lo, hi = boot_ci(yddi.numpy()[m], mprob[m])
    print('  {0:5s} n={1:3d}  AUROC={2:.3f}  [{3:.3f}, {4:.3f}]'.format(grp, int(m.sum()), a, lo, hi))

In [ ]:
# --- 5. Training (ERM vs Group-DRO) ---
HID = 16

def init(seed, D):
    g = torch.Generator().manual_seed(seed)
    return [(torch.randn(D, HID, generator=g) * 0.3).requires_grad_(True), torch.zeros(HID, requires_grad=True),
            (torch.randn(HID, 1, generator=g) * 0.3).requires_grad_(True), torch.zeros(1, requires_grad=True)]

def train_head(Xtr, ytr, gtr, mode, seed=0, iters=1500, lr=0.05, eta=1.0):
    P = init(seed, Xtr.shape[1]); opt = torch.optim.Adam(P, lr=lr)
    bce = torch.nn.functional.binary_cross_entropy_with_logits
    pw = torch.tensor(max(float((ytr == 0).sum()), 1.0) / max(float((ytr == 1).sum()), 1.0))
    gm = [torch.tensor(gtr == grp) for grp in GROUPS]; q = torch.ones(3) / 3
    for it in range(iters):
        per = bce(fwd(Xtr, P), ytr, pos_weight=pw, reduction='none')
        if mode == 'erm':
            loss = per.mean()
        else:
            lg = torch.stack([per[m].mean() if m.any() else torch.tensor(0.0) for m in gm])
            q = q * torch.exp(eta * lg.detach()); q = q / q.sum(); loss = (q * lg).sum()
        opt.zero_grad(); loss.backward(); opt.step()
    return P

def strat_split(g, y, frac=0.6, seed=0):
    rng = np.random.default_rng(seed); tr = np.zeros(len(g), bool); yy = y.numpy()
    for grp in GROUPS:
        for lab in (0.0, 1.0):
            sel = np.where((g == grp) & (yy == lab))[0]; rng.shuffle(sel)
            tr[sel[:int(round(frac * len(sel)))]] = True
    return tr

In [ ]:
# --- 6. DDI-internal fairness: zero-shot vs transferred concepts (CIs) ---
trd = strat_split(gddi, yddi); ted = ~trd
SEEDS = [0, 1, 2]

def run_src(name, C):
    Xtr, ytr, gtr = C[trd], yddi[trd], gddi[trd]
    Xte, yte, gte = C[ted], yddi[ted], gddi[ted]
    for mode in ['erm', 'gdro']:
        probs = np.zeros(len(yte)); mono = []
        for s in SEEDS:
            P = train_head(Xtr, ytr, gtr, mode, seed=s)
            probs += torch.sigmoid(fwd(Xte, P)).detach().numpy() / len(SEEDS)
            mono.append(monotonicity(P, Xte))
        aucs = {}
        for grp in GROUPS:
            m = gte == grp
            aucs[grp] = boot_ci(yte.numpy()[m], probs[m])
        worst = min(aucs[g][0] for g in GROUPS)
        print('  [{0:8s}/{1:4s}] mono={2:.3f}  worst_auc={3:.3f}'.format(name, mode, float(np.mean(mono)), worst))
        for grp in GROUPS:
            a, lo, hi = aucs[grp]
            print('     {0:5s} n={1:3d}  AUROC={2:.3f}  [{3:.3f}, {4:.3f}]'.format(grp, int((gte == grp).sum()), a, lo, hi))

print('DDI-internal fairness, test n={0} (groups {1}):'.format(int(ted.sum()), {g: int((gddi[ted] == g).sum()) for g in GROUPS}))
run_src('zeroshot', Czs)
run_src('transfer', Cxfer)

## How to read this

- **Cell 3** tells you whether the transfer probe is even valid on Derm7pt (concept AUROC) before trusting it on DDI.
- **Cell 4 (anchor)** is the robust headline: DermLIP zero-shot malignancy detection by skin tone, with bootstrap 95% CIs. If the dark CI sits clearly below the light CI, the disparity is real (not noise).
- **Cell 6** compares zero-shot vs transferred concepts as the interpretable bottleneck, and ERM vs Group-DRO, per skin tone with CIs. Overlapping CIs => differences are within noise (DDI is small).

If transferred concepts raise the bottleneck AUROC and monotonicity over zero-shot, that justifies the transfer step. Either way, the CIs keep us honest about what DDI's sample size can and cannot show -- and motivate the powered Fitzpatrick17k -> DDI study once access arrives.